In [1]:
!pip uninstall -y torchao
!pip install -q "transformers>=5.17,<6" "peft>=0.21" rapidfuzz

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [2]:
%%writefile /kaggle/working/fl_eval_standalone.py
"""Standalone evaluation of the final federated (IFCA + FedProx) LoRA adapters on the valid and test splits.

Needs only: fl_state_latest.pt, the Tamil IndicVoices subset (manifest.parquet + audio/), internet for openai/whisper-small.
Decoding matches the centralized run: Tamil / transcribe, greedy, 395-token cap. References are the corpus
`normalized` transcript (fallback to `text`), exactly as in the training scripts. No training happens here.
"""
import argparse, json, math, os, time
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
from torch.utils.data import DataLoader, Dataset
from rapidfuzz.distance import Levenshtein
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, set_peft_model_state_dict

LANGUAGE, LANGUAGE_NAME, TASK = "ta", "Tamil", "transcribe"
TARGET_SR, MAX_LABEL_LEN = 16000, 448
DUR_BINS = [0, 2, 4, 6, 8, 12, 16, 20, 30, np.inf]
DUR_LABELS = ["0-2s", "2-4s", "4-6s", "6-8s", "8-12s", "12-16s", "16-20s", "20-30s", "30s+"]


def log(msg):
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}", flush=True)


# ---------------------------------------------------------------- metrics (same as the training scripts)
def edit_counts(ref_tokens, hyp_tokens):
    s = d = i = 0
    for op in Levenshtein.editops(ref_tokens, hyp_tokens):
        if op.tag == "replace":
            s += 1
        elif op.tag == "delete":
            d += 1
        else:
            i += 1
    return s, d, i


def utterance_stats(ref, hyp):
    rw, hw = ref.split(), hyp.split()
    ws, wd, wi = edit_counts(rw, hw)
    rc, hc = ref.strip(), hyp.strip()
    cs, cd, ci = edit_counts(rc, hc)
    return {"w_sub": ws, "w_del": wd, "w_ins": wi, "w_n": len(rw), "hyp_words": len(hw),
            "c_sub": cs, "c_del": cd, "c_ins": ci, "c_n": len(rc)}


def add_derived(df):
    df = df.copy()
    df["w_err"] = df.w_sub + df.w_del + df.w_ins
    df["c_err"] = df.c_sub + df.c_del + df.c_ins
    df["utt_wer"] = df.w_err / df.w_n.clip(lower=1)
    df["utt_cer"] = df.c_err / df.c_n.clip(lower=1)
    return df


def agg_metrics(g):
    wn, cn = max(g.w_n.sum(), 1), max(g.c_n.sum(), 1)
    return pd.Series({
        "n_utts": len(g), "hours": g.duration.sum() / 3600.0,
        "WER": 100.0 * g.w_err.sum() / wn, "CER": 100.0 * g.c_err.sum() / cn,
        "sub_pct": 100.0 * g.w_sub.sum() / wn, "del_pct": 100.0 * g.w_del.sum() / wn,
        "ins_pct": 100.0 * g.w_ins.sum() / wn,
        "exact_match_pct": 100.0 * (g.w_err == 0).mean(),
        "median_utt_WER": 100.0 * g.utt_wer.median(),
        "len_ratio_hyp_over_ref": g.hyp_words.sum() / wn,
        "hit_length_cap_pct": 100.0 * g.hit_cap.mean(),
    })


# ---------------------------------------------------------------- data
def load_audio(path):
    try:
        audio, sr = sf.read(path, dtype="float32", always_2d=False)
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        if sr != TARGET_SR:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=TARGET_SR)
    except Exception:
        audio, _ = librosa.load(path, sr=TARGET_SR, mono=True)
    if audio.size == 0 or not np.isfinite(audio).all():
        raise ValueError("empty or non-finite audio")
    return audio


class EvalAudioDataset(Dataset):
    def __init__(self, df, processor):
        self.paths = df["audio_full_path"].tolist()
        self.fe = processor.feature_extractor

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        try:
            feats = self.fe(load_audio(self.paths[idx]), sampling_rate=TARGET_SR).input_features[0]
            return {"input_features": feats, "idx": idx, "ok": True}
        except Exception:
            feats = self.fe(np.zeros(TARGET_SR, dtype=np.float32), sampling_rate=TARGET_SR).input_features[0]
            return {"input_features": feats, "idx": idx, "ok": False}


def collate(batch):
    return {"input_features": torch.from_numpy(np.stack([b["input_features"] for b in batch])),
            "idx": [b["idx"] for b in batch], "ok": [b["ok"] for b in batch]}


# ---------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--ckpt", default="/kaggle/input/models/ajeffreyrufus/fl-latest-model/pytorch/default/1/fl_state_latest.pt")
    ap.add_argument("--data-root", default="/kaggle/input/datasets/ajeffreyrufus/tamil-indicvoices/tamil_fl_data")
    ap.add_argument("--out-dir", default="/kaggle/working/fl_eval")
    ap.add_argument("--model", default="openai/whisper-small")
    ap.add_argument("--batch-size", type=int, default=16)
    ap.add_argument("--gen-cap", type=int, default=395)
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--splits", nargs="+", default=["valid", "test"])
    ap.add_argument("--limit", type=int, default=0, help="debug: evaluate only the first N utterances per split")
    a = ap.parse_args()

    out = Path(a.out_dir); out.mkdir(parents=True, exist_ok=True)
    assert torch.cuda.is_available(), "Turn on a GPU accelerator."
    device = "cuda"
    amp = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    log(f"GPU {torch.cuda.get_device_name(0)} | autocast {amp}")

    # --- manifest, labels and the same length filter as the training scripts (drops the 67 over-long rows)
    manifest = pd.read_parquet(Path(a.data_root) / "manifest.parquet")
    manifest["audio_full_path"] = manifest["audio_path"].apply(lambda p: str(Path(a.data_root) / p))
    manifest["label_text"] = manifest["normalized"].where(manifest["normalized"].astype(str).str.strip() != "", manifest["text"])
    manifest = manifest[manifest["label_text"].astype(str).str.strip() != ""].reset_index(drop=True)
    processor = WhisperProcessor.from_pretrained(a.model, language=LANGUAGE_NAME, task=TASK)
    texts, lens = manifest["label_text"].tolist(), []
    for s in range(0, len(texts), 2000):
        lens.extend(len(ids) for ids in processor.tokenizer(texts[s:s + 2000]).input_ids)
    before = len(manifest)
    manifest = manifest[np.array(lens) <= MAX_LABEL_LEN].reset_index(drop=True)
    log(f"Label-length filter dropped {before - len(manifest)} of {before} rows "
        f"| valid={int((manifest.split == 'valid').sum())} test={int((manifest.split == 'test').sum())}")

    # --- model + federated checkpoint (config identical to training: r=32, alpha=64, dropout 0.05, q/v)
    payload = torch.load(a.ckpt, map_location="cpu", weights_only=False)
    cluster_states, client_cluster = payload["cluster_states"], payload["client_cluster"]
    log(f"Checkpoint: next_round={payload.get('next_round')} clusters={sorted(cluster_states)} "
        f"district->cluster={dict(sorted(client_cluster.items()))}")
    model = WhisperForConditionalGeneration.from_pretrained(a.model)
    model.generation_config.language, model.generation_config.task = LANGUAGE, TASK
    model.generation_config.forced_decoder_ids = None
    model = get_peft_model(model, LoraConfig(r=32, lora_alpha=64, target_modules=["q_proj", "v_proj"],
                                             lora_dropout=0.05, bias="none"))
    model.to(device).eval()
    eos = processor.tokenizer.eos_token_id

    summary, overall_rows = {}, {}
    for split in a.splits:
        df = manifest[manifest["split"] == split].reset_index(drop=True)
        if a.limit:
            df = df.head(a.limit)
        log(f"Transcribing {split}: {len(df)} utterances")
        rows, t0 = [], time.time()
        for district, group in df.groupby("district", dropna=False):
            cluster = int(client_cluster.get(str(district), 0))
            set_peft_model_state_dict(model, cluster_states[cluster])
            group = group.reset_index(drop=True)
            loader = DataLoader(EvalAudioDataset(group, processor), batch_size=a.batch_size, shuffle=False,
                                num_workers=a.workers, collate_fn=collate, pin_memory=True)
            for bi, batch in enumerate(loader):
                feats = batch["input_features"].to(device, non_blocking=True)
                with torch.no_grad(), torch.autocast("cuda", dtype=amp):
                    gen = model.generate(input_features=feats, language=LANGUAGE, task=TASK,
                                         max_new_tokens=a.gen_cap, num_beams=1, do_sample=False)
                hit = (gen[:, -1] != eos).cpu().tolist()
                hyps = [h.strip() for h in processor.tokenizer.batch_decode(gen, skip_special_tokens=True)]
                for k, (idx, ok, hyp) in enumerate(zip(batch["idx"], batch["ok"], hyps)):
                    item = group.iloc[idx]
                    stats = utterance_stats(item.label_text, hyp) if ok else \
                        {c: np.nan for c in ("w_sub", "w_del", "w_ins", "w_n", "hyp_words", "c_sub", "c_del", "c_ins", "c_n")}
                    rows.append({"split": split, "audio_path": item.audio_path, "district": item.district,
                                 "duration": item.duration, "ref": item.label_text, "hyp": hyp,
                                 "cluster": cluster, "hit_cap": hit[k], "audio_ok": ok, **stats})
                if bi % 50 == 0:
                    log(f"  [{split}] {district}: batch {bi + 1}/{len(loader)} | total {len(rows)}/{len(df)} "
                        f"| {(time.time() - t0) / 60:.1f} min")
        preds = pd.DataFrame(rows)
        preds.to_csv(out / f"predictions_{split}.csv", index=False, encoding="utf-8-sig")
        good = add_derived(preds[preds["audio_ok"]].copy())
        overall_rows[split] = agg_metrics(good)
        good.groupby("district").apply(agg_metrics).sort_values("WER").to_csv(out / f"per_district_{split}.csv")
        b = good.copy(); b["dur_bucket"] = pd.cut(b.duration, DUR_BINS, labels=DUR_LABELS, right=False)
        b.groupby("dur_bucket", observed=True).apply(agg_metrics).to_csv(out / f"per_duration_{split}.csv")
        good.groupby("cluster").apply(agg_metrics).to_csv(out / f"per_cluster_{split}.csv")
        pd.DataFrame(overall_rows).T.to_csv(out / "overall_by_split.csv")
        log(f"{split}: WER {overall_rows[split]['WER']:.2f} | CER {overall_rows[split]['CER']:.2f} "
            f"| cap-hit {overall_rows[split]['hit_length_cap_pct']:.2f}% | {(time.time() - t0) / 60:.1f} min")
    log("DONE. Results in " + str(out))


if __name__ == "__main__":
    main()

Writing /kaggle/working/fl_eval_standalone.py


In [3]:
!python /kaggle/working/fl_eval_standalone.py

[2026-11-01 14:30:57] GPU Tesla T4 | autocast torch.bfloat16
preprocessor_config.json: 100%|███████████████| 185k/185k [00:00<00:00, 258MB/s]
config.json: 100%|█████████████████████████| 1.97k/1.97k [00:00<00:00, 9.35MB/s]
tokenizer_config.json: 100%|██████████████████| 283k/283k [00:00<00:00, 379MB/s]
vocab.json: 100%|████████████████████████████| 836k/836k [00:00<00:00, 20.6MB/s]
tokenizer.json: 100%|███████████████████████| 2.48M/2.48M [00:00<00:00, 136MB/s]
merges.txt: 100%|█████████████████████████████| 494k/494k [00:00<00:00, 103MB/s]
normalizer.json: 100%|██████████████████████| 52.7k/52.7k [00:00<00:00, 103MB/s]
added_tokens.json: 100%|████████████████████| 34.6k/34.6k [00:00<00:00, 100MB/s]
special_tokens_map.json: 100%|█████████████| 2.19k/2.19k [00:00<00:00, 10.7MB/s]
[2026-11-01 14:31:09] Label-length filter dropped 67 of 62917 rows | valid=8862 test=9969
[2026-11-01 14:31:10] Checkpoint: next_round=41 clusters=[0, 1, 2] district->cluster={'Cuddalore': 2, 'Dharmapuri': 2, '

In [4]:
import pandas as pd
import shutil
import os

E = "/kaggle/working/fl_eval"

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)

print("OVERALL RESULTS")
print(pd.read_csv(f"{E}/overall_by_split.csv").round(2).to_string(index=False))

for f in ["per_cluster_test", "per_district_test", "per_duration_test"]:
    print(f"\n{f.upper()}")
    print(pd.read_csv(f"{E}/{f}.csv").iloc[:, :9].round(2).to_string(index=False))

shutil.make_archive("/kaggle/working/fl_eval_results", "zip", E)

OVERALL RESULTS
Unnamed: 0  n_utts  hours   WER   CER  sub_pct  del_pct  ins_pct  exact_match_pct  median_utt_WER  len_ratio_hyp_over_ref  hit_length_cap_pct
     valid  8862.0  21.15 63.42 22.41    52.00     6.09     5.32             1.11           62.96                    0.99                6.43
      test  9969.0  24.08 60.87 21.31    50.35     5.90     4.62             1.18           60.87                    0.99                6.42

PER_CLUSTER_TEST
 cluster  n_utts  hours   WER   CER  sub_pct  del_pct  ins_pct  exact_match_pct
       2  9969.0  24.08 60.87 21.31    50.35      5.9     4.62             1.18

PER_DISTRICT_TEST
      district  n_utts  hours   WER   CER  sub_pct  del_pct  ins_pct  exact_match_pct
   Krishnagiri  1058.0   2.48 56.48 18.09    46.88     5.02     4.57             2.17
     Cuddalore   826.0   2.05 58.76 19.68    47.85     5.59     5.33             0.73
         Erode   800.0   1.89 58.82 19.17    48.91     5.90     4.01             1.88
Mayiladuthurai  1

'/kaggle/working/fl_eval_results.zip'